# Phase 5: edge-aware NEDI–IMDN fusion validation

The first fusion test mixed NEDI into every pixel and selected 100% IMDN. This second test is more targeted:

- IMDN remains unchanged outside strong edges.
- A small amount of NEDI is added only around detected edges.
- Edges are detected from the LR input, never from the reference HR image.
- The same five DIV2K validation crops and all three scales are used.
- IMDN alone is included as the baseline that the edge-aware method must beat.

This notebook reuses the reconstructions cached by notebook 14. It does not rerun NEDI or IMDN and does not need a GPU.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/divinesta/SuperResolution-Comparative-Analysis.git'
REPO_ROOT = Path('/content/SuperResolution-Comparative-Analysis')
if REPO_ROOT.exists():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_ROOT / 'requirements.txt')],
    check=True,
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('Repository and dependencies ready.')


In [ ]:
DATA_ROOT = Path('/content/drive/MyDrive/FYP_SR_Data')
WEIGHTED_RUN_ROOT = DATA_ROOT / 'results' / 'phase5' / 'weight_selection_global_v1'
CACHE_ROOT = WEIGHTED_RUN_ROOT / 'reconstruction_cache'
OUTPUT_ROOT = DATA_ROOT / 'results' / 'phase5' / 'edge_aware_validation_v1'
METRICS_ROOT = OUTPUT_ROOT / 'metrics'
FIGURE_ROOT = OUTPUT_ROOT / 'figures'
METRICS_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

VALIDATION_IDS = ('0801', '0802', '0803', '0804', '0805')
SCALES = (2, 3, 4)
print('Input cache:', CACHE_ROOT)
print('Output folder:', OUTPUT_ROOT)


## Settings being tested

Three edge cut-offs are used:

- 80th percentile: more pixels are treated as edges.
- 90th percentile: only stronger edges are used.
- 95th percentile: only the strongest edges are used.

At those edges, the NEDI contribution is tested at 5%, 10%, 20%, and 30%. Everywhere else remains 100% IMDN. This gives 12 edge-aware settings plus IMDN alone.


In [ ]:
from app.fusion.edge_aware import (
    DEFAULT_EDGE_NEDI_WEIGHTS,
    DEFAULT_EDGE_PERCENTILES,
)

EDGE_PERCENTILES = DEFAULT_EDGE_PERCENTILES
EDGE_NEDI_WEIGHTS = DEFAULT_EDGE_NEDI_WEIGHTS
DILATION_RADIUS = 1
print('Edge percentiles:', EDGE_PERCENTILES)
print('NEDI percentages at detected edges:', [f'{weight:.0%}' for weight in EDGE_NEDI_WEIGHTS])
print('Edge expansion radius:', DILATION_RADIUS, 'pixel')


## Load the cached validation images

This cell stops clearly if notebook 14's cached files are missing.


In [ ]:
from app.evaluation.images import load_rgb_image

cases = []
required_names = ('reference_hr.png', 'input_lr.png', 'nedi.png', 'imdn.png')
for scale in SCALES:
    for image_id in VALIDATION_IDS:
        case_root = CACHE_ROOT / f'x{scale}' / image_id
        missing = [name for name in required_names if not (case_root / name).is_file()]
        if missing:
            raise FileNotFoundError(
                f'Missing cached files for {image_id} x{scale}: {missing}. '
                'Run notebook 14 completely first.'
            )
        reference_hr = load_rgb_image(case_root / 'reference_hr.png')
        input_lr = load_rgb_image(case_root / 'input_lr.png')
        nedi = load_rgb_image(case_root / 'nedi.png')
        imdn = load_rgb_image(case_root / 'imdn.png')
        if reference_hr.size != nedi.size or reference_hr.size != imdn.size:
            raise ValueError(f'Cached output sizes do not match for {image_id} x{scale}.')
        cases.append((image_id, scale, reference_hr, input_lr, nedi, imdn))

if len(cases) != 15:
    raise RuntimeError(f'Expected 15 validation cases; loaded {len(cases)}.')
print('Loaded all', len(cases), 'cached validation cases.')


## Evaluate every edge-aware setting

The main selection value remains mean PSNR-Y. Mean SSIM-Y is the tie-break. No setting is selected by looking at the final benchmark datasets.


In [ ]:
from app.evaluation.experiment import write_results_csv
from app.fusion.edge_aware import evaluate_edge_aware_grid

all_records = []
for image_id, scale, reference_hr, input_lr, nedi, imdn in cases:
    case_records = evaluate_edge_aware_grid(
        reference_hr,
        input_lr,
        nedi,
        imdn,
        scale,
        edge_percentiles=EDGE_PERCENTILES,
        edge_nedi_weights=EDGE_NEDI_WEIGHTS,
        dilation_radius=DILATION_RADIUS,
    )
    for record in case_records:
        all_records.append({
            'validation_dataset': 'DIV2K_valid',
            'image': f'{image_id}.png',
            'scale': f'x{scale}',
            'method': 'edge_aware_nedi_imdn_fusion',
            'edge_guide': 'bicubic_upsampled_lr_luminance_sobel',
            'degradation': 'project_bicubic_downsampling',
            'metric_border_pixels': scale,
            **record,
        })
    print(f'Complete: {image_id}.png x{scale}')

expected_rows = len(cases) * (1 + len(EDGE_PERCENTILES) * len(EDGE_NEDI_WEIGHTS))
if len(all_records) != expected_rows:
    raise RuntimeError(f'Expected {expected_rows} rows; produced {len(all_records)}.')
all_csv = write_results_csv(
    all_records,
    METRICS_ROOT / 'edge_aware_validation_all_configs.csv',
    overwrite=True,
)
print('Saved', len(all_records), 'rows to', all_csv)


In [ ]:
import json
from datetime import UTC, datetime

from app.fusion.edge_aware import (
    select_best_edge_aware_config,
    summarise_edge_aware_results,
)

summary_records = summarise_edge_aware_results(all_records)
baseline = next(row for row in summary_records if row['config_id'] == 'imdn_only')
for row in summary_records:
    row['psnr_y_change_vs_imdn'] = row['mean_psnr_y'] - baseline['mean_psnr_y']
    row['ssim_y_change_vs_imdn'] = row['mean_ssim_y'] - baseline['mean_ssim_y']
summary_csv = write_results_csv(
    summary_records,
    METRICS_ROOT / 'edge_aware_config_summary.csv',
    overwrite=True,
)
selected = select_best_edge_aware_config(summary_records)
improved_over_imdn = selected['config_id'] != 'imdn_only' and selected['mean_psnr_y'] > baseline['mean_psnr_y']
selection_record = {
    'method': 'edge_aware_nedi_imdn_fusion',
    'selected_config_id': selected['config_id'],
    'improved_over_imdn': improved_over_imdn,
    'selected_edge_percentile': selected['edge_percentile'],
    'selected_nedi_weight_at_edges': selected['edge_nedi_weight'],
    'selected_imdn_weight_at_edges': selected['edge_imdn_weight'],
    'selected_mean_edge_pixel_fraction': selected['mean_edge_pixel_fraction'],
    'selected_mean_psnr_y': selected['mean_psnr_y'],
    'selected_mean_ssim_y': selected['mean_ssim_y'],
    'imdn_only_mean_psnr_y': baseline['mean_psnr_y'],
    'imdn_only_mean_ssim_y': baseline['mean_ssim_y'],
    'psnr_y_change_vs_imdn': selected['mean_psnr_y'] - baseline['mean_psnr_y'],
    'ssim_y_change_vs_imdn': selected['mean_ssim_y'] - baseline['mean_ssim_y'],
    'selection_rule': 'highest_mean_psnr_y_then_mean_ssim_y_then_less_nedi',
    'validation_dataset': 'DIV2K_valid',
    'validation_image_ids': list(VALIDATION_IDS),
    'validation_scales': list(SCALES),
    'edge_percentile_candidates': list(EDGE_PERCENTILES),
    'edge_nedi_weight_candidates': list(EDGE_NEDI_WEIGHTS),
    'dilation_radius': DILATION_RADIUS,
    'generated_at_utc': datetime.now(UTC).isoformat(),
}
selection_json = METRICS_ROOT / 'edge_aware_selected_config.json'
selection_json.write_text(json.dumps(selection_record, indent=2) + '\n', encoding='utf-8')

print('\nSELECTED:', selected['config_id'])
print('Improved over IMDN:', improved_over_imdn)
print(f"PSNR-Y change versus IMDN: {selection_record['psnr_y_change_vs_imdn']:+.6f} dB")
print(f"SSIM-Y change versus IMDN: {selection_record['ssim_y_change_vs_imdn']:+.6f}")
print('Saved:', summary_csv)
print('Saved:', selection_json)


In [ ]:
import matplotlib.pyplot as plt

ordered = sorted(
    summary_records,
    key=lambda row: float(row['psnr_y_change_vs_imdn']),
    reverse=True,
)
labels = [row['config_id'] for row in ordered]
changes = [float(row['psnr_y_change_vs_imdn']) for row in ordered]
colours = ['#1769aa' if change > 0 else '#c44e52' if change < 0 else '#777777' for change in changes]

figure, axis = plt.subplots(figsize=(10, 6))
axis.barh(range(len(labels)), changes, color=colours)
axis.set_yticks(range(len(labels)), labels=labels)
axis.invert_yaxis()
axis.axvline(0, color='black', linewidth=1)
axis.set_xlabel('Mean PSNR-Y change compared with IMDN alone (dB)')
axis.set_title('Phase 5 edge-aware fusion validation')
axis.grid(axis='x', alpha=0.25)
figure.tight_layout()
figure_path = FIGURE_ROOT / 'edge_aware_psnr_change_vs_imdn.png'
figure.savefig(figure_path, dpi=180, bbox_inches='tight')
plt.show()
print('Saved:', figure_path)


## Return the results

Download and send back these three files from **MyDrive/FYP_SR_Data/results/phase5/edge_aware_validation_v1/metrics/**:

1. **edge_aware_validation_all_configs.csv**
2. **edge_aware_config_summary.csv**
3. **edge_aware_selected_config.json**

If an edge-aware setting beats IMDN, we will lock it and run the final benchmark. If IMDN alone still wins, we will stop fusion experiments and report the limitation honestly.
